# LangChain Hands-on 12 — Tools & Tool Calling
**Trainer:** Ajeetkumar

**What we will practise**
- What is a tool?
- Create tools with `@tool`
- StructuredTool & Pydantic args
- Built-in tools: Wikipedia, DuckDuckGo search
- bind_tools — LLM asks for a tool
- Run the tool and send the result back (manual loop)

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq langchain-community wikipedia ddgs

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# What is a Tool?
- A **Python function** the LLM can ask to use: calculator, search, database, API, email …
- The LLM **does not run** the tool. It only says *"call tool X with these arguments"*. Our code runs it.

```
User ─▶ LLM ─▶ "call get_weather(city='Pune')" ─▶ our code runs it ─▶ result ─▶ LLM ─▶ final answer
```

## 1. Create a tool with @tool

In [ ]:
from langchain.tools import tool

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [ ]:
multiply.name, multiply.description, multiply.args

In [ ]:
multiply.invoke({"a": 12, "b": 8})

⚠️ The **docstring** and **type hints** are very important — the LLM reads them to decide when and how to use the tool.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather of an Indian city."""
    data = {"pune": "28°C, cloudy", "mumbai": "31°C, humid", "delhi": "35°C, sunny"}
    return data.get(city.lower(), "25°C, clear")

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between INR, USD and EUR."""
    rate_to_inr = {"INR": 1, "USD": 84, "EUR": 91}
    value = amount * rate_to_inr[from_currency.upper()] / rate_to_inr[to_currency.upper()]
    return f"{amount} {from_currency} = {round(value, 2)} {to_currency}"

In [ ]:
get_weather.invoke({"city": "Pune"})

## 2. Tool with Pydantic argument schema (better validation)

In [ ]:
from pydantic import BaseModel, Field

class EMIInput(BaseModel):
    principal: float = Field(description="loan amount in INR")
    annual_rate: float = Field(description="interest rate per year in percent")
    years: int = Field(description="loan tenure in years")

@tool(args_schema=EMIInput)
def emi_calculator(principal: float, annual_rate: float, years: int) -> str:
    """Calculate the monthly EMI of a loan."""
    r = annual_rate / 12 / 100
    n = years * 12
    emi = principal * r * (1 + r) ** n / ((1 + r) ** n - 1)
    return f"Monthly EMI = Rs {round(emi)}"

In [ ]:
emi_calculator.invoke({"principal": 500000, "annual_rate": 9, "years": 5})

## 3. StructuredTool — make a tool from an existing function

In [ ]:
from langchain_core.tools import StructuredTool

def add(a: int, b: int) -> int:
    return a + b

add_tool = StructuredTool.from_function(func=add, name="add", description="Add two numbers")
add_tool.invoke({"a": 5, "b": 7})

## 4. Built-in community tools

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500))
print(wiki.invoke("Chhatrapati Shivaji Maharaj"))

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

In [ ]:
search = DuckDuckGoSearchRun()
print(search.invoke("latest LangChain version"))

Other popular tools: **Tavily** search (`langchain-tavily`), Python REPL, SQL database toolkit, Gmail, GitHub, Google Calendar, any REST API.

## 5. bind_tools — give tools to the LLM

In [ ]:
tools = [multiply, get_weather, convert_currency, emi_calculator]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
ai_msg = llm_with_tools.invoke("What is the weather in Mumbai?")

In [ ]:
ai_msg.content        # usually empty

In [ ]:
ai_msg.tool_calls     # 👈 the LLM is asking us to run a tool

In [ ]:
llm_with_tools.invoke("Hi, how are you?").tool_calls     # no tool needed → empty list

## 6. Run the tool and send the result back (manual loop)

In [ ]:
from langchain.messages import HumanMessage, ToolMessage

In [ ]:
tools_by_name = {t.name: t for t in tools}

messages = [HumanMessage("What is 1500 USD in INR, and what's the EMI for that INR amount at 9% for 3 years?")]

In [ ]:
for step in range(5):
    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)
    if not ai_msg.tool_calls:          # no more tools → final answer
        break
    for call in ai_msg.tool_calls:
        result = tools_by_name[call["name"]].invoke(call["args"])
        print(f"🔧 {call['name']}({call['args']}) → {result}")
        messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

In [ ]:
print(ai_msg.content)

> This loop (LLM → tool → LLM → …) is exactly what an **Agent** does automatically. Next notebook!

# Tool design tips
- One clear job per tool, clear name
- Good docstring: *when* to use it
- Typed arguments (str, int, float, Literal…)
- Return short text/JSON; handle errors inside and return a message

## 🧪 Practice
1. Make a `get_stock_price(symbol)` tool with fake data and ask "Compare TCS and Infosys".
2. Make a `send_email(to, subject, body)` tool that only prints.
3. Use the Wikipedia tool + LLM to write a 5-line bio of any scientist.